# FlowRec — Results Analysis

This notebook visualises experiment results from `results/`. It does not contain pipeline logic.
All outputs must already exist (run the full pipeline first).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
RESULTS = Path('../results')
FIG_DIR = RESULTS / 'figures'
FIG_DIR.mkdir(exist_ok=True)

## 1. Overall Evaluation

In [ ]:
eval_df = pd.read_csv(RESULTS / 'final_eval.csv')
pivot = eval_df.pivot(index='system', columns='metric', values='value')
display(pivot.style.format('{:.4f}').highlight_max(axis=0, color='#d4edda'))

In [ ]:
plot_metrics = ['recall@10', 'ndcg@10', 'mrr']
plot_data = eval_df[eval_df['metric'].isin(plot_metrics)]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(12, 4), sharey=False)
for ax, metric in zip(axes, plot_metrics):
    subset = plot_data[plot_data['metric'] == metric].sort_values('value')
    ax.barh(subset['system'], subset['value'])
    ax.set_title(metric)
    ax.set_xlabel('Score')
plt.suptitle('System Comparison', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'system_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Slice Analysis

In [ ]:
slice_df = pd.read_csv(RESULTS / 'slice_eval_combined.csv')

for slice_type in slice_df['slice_type'].unique():
    subset = slice_df[
        (slice_df['slice_type'] == slice_type) &
        (slice_df['metric'] == 'recall@10')
    ]
    pivot = subset.pivot(index='slice_value', columns='system', values='value')
    
    fig, ax = plt.subplots(figsize=(7, 3.5))
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'Recall@10 by {slice_type}')
    ax.set_xlabel('')
    ax.set_ylabel('Recall@10')
    ax.legend(title='System', bbox_to_anchor=(1, 1))
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'slice_{slice_type}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Feature Importance

In [ ]:
imp = pd.read_csv(RESULTS / 'feature_importance.csv').head(15)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp['feature'][::-1], imp['importance'][::-1])
ax.set_title('Top 15 Features by Gain')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Target Rank Distribution

In [ ]:
import os

systems = ['popularity', 'cooccurrence', 'cooc_ranker']
fig, axes = plt.subplots(1, len(systems), figsize=(14, 4), sharey=True)

for ax, system in zip(axes, systems):
    fpath = RESULTS / f'error_report_{system}.csv'
    if not fpath.exists():
        ax.set_title(f'{system}\n(no data)')
        continue
    df = pd.read_csv(fpath)
    hits = df[df['target_rank'] > 0]['target_rank']
    ax.hist(hits, bins=30, edgecolor='white', linewidth=0.5)
    ax.set_title(system.replace('_', '+'))
    ax.set_xlabel('Target rank')

axes[0].set_ylabel('Sessions')
plt.suptitle('Distribution of Target Item Rank', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'rank_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Coverage

In [ ]:
cov = pd.read_csv(RESULTS / 'coverage_stats.csv')
display(cov)